## Make a figure showing all GHG data 
#### Including a fit a curve with the NOAA method
#### Including a seasonal cycle


The NOAA method for curve fitting used is described here: 

The method is described: https://gml.noaa.gov/ccgg/mbl/crvfit/crvfit.html 

The code is available at: https://gml.noaa.gov/aftp/user/thoning/ccgcrv/

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot

import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)
    

from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
import process_data
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload


from input.read_wdc_data import AvailableData, create_data_reader
#save figures in...
dir_save = 'output/ghg_ts/'

### First, read in all the GHG data

In [ ]:
## Read all data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


#### Remove outlies (e.g. CO peak)

In [ ]:
sel_species = ds_all.species
unique_species = np.unique(sel_species)
species_sel = unique_species

In [ ]:
ds_all_rem_out = process_data.rem_out(ds_all, std_fac=10, z_threshold=4)
## plot removed data
var = "value"

for s in species_sel:

    f, axs = plt.subplots(2, 1, sharex=True)
    plt.suptitle("Removed outliers")
    ds_all.sel(dataset=s)[var].plot(ls="", marker="o", ax=axs[0])
    ds_all.sel(dataset=s)[var + "_unc"].plot(ls="", marker="o", ax=axs[1])
    # new data:
    ds_all_rem_out.sel(dataset=s)[var].plot(ls="", marker=".", ax=axs[0])
    ds_all_rem_out.sel(dataset=s)[var + "_unc"].plot(ls="", marker=".", ax=axs[1])
    plt.show()

#### Prepare the figure

In [ ]:
## Define the data periods I want to compare
t0 = ds_all.isel(time=0)
t1 = ds_all.isel(time=-1)
print(f"Total time period with data:: {t0.time.values} to {t1.time.values}")

compare_periods = {
    'A': (t1.time.values, dt.datetime(2006,12,31)), #old and flask data
    'B': (dt.datetime(2008,1,1), dt.datetime(2011,12,31)), #only flask data
    'C': (dt.datetime(2012,1,1), t1.time.values), # only new data
    'D': (dt.datetime(2014,1,1), t1.time.values) # only ozone data
}

In [ ]:
# Define fit paramaters for the curve fitting
# Default values
fit_params_defaults = {'shortterm': 80,
                'longterm': 667,
                'numpolyterms': 2, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}

fit_properties = {}
for dataset in ds_all.dataset:
    dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset_name] = fit_params_defaults.copy()
    
# Update fit_properties with non-default values:
# Remove sampleinterval for flask data:
for dataset in fit_properties:
    if '_flask' in dataset:
        fit_properties[dataset]['sampleinterval'] = 0 #if 0, determine from xp (time)

#fit_properties['CO']['numpolyterms'] = 3


In [ ]:
import run_curve_fit

%autoreload 2

for period, (start_date, end_date) in compare_periods.items():
    # Filter the data for the current period
    period_data = ds_all.sel(time=slice(start_date, end_date))

    # Define figure properties for this period
    # fig_properties = {
    #'color': 'w',
    # }

    for dataset in period_data.dataset:
        # Get the data for the current dataset and period
        data = period_data.sel(dataset=dataset)["value"]

        # Define fit properties for this dataset

        # Perform the curve fitting and obtain the interpolated time axis
        filt = run_curve_fit.run_ccgfilter(
            ds=data,
            dataset_str=dataset.item(),
            t1=start_date,
            t2=end_date,
            **fit_properties[dataset.item()]
        )

In [ ]:
dataset.item()

In [ ]:
filt